In [1]:
import cv2
import mediapipe as mp
import numpy as np
import winsound  

In [2]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=1,refine_landmarks=True)


In [3]:
LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
EAR_THRESHOLD = 0.15
closed_frames = 0

In [4]:
def euclidean_dist(p1, p2):
    return np.linalg.norm(p1 - p2)

def eye_aspect_ratio(eye):
    A = euclidean_dist(eye[1], eye[5])
    B = euclidean_dist(eye[2], eye[4])
    C = euclidean_dist(eye[0], eye[3])

    return (A + B) / (2.0 * C)

In [ ]:
cap = cv2.VideoCapture("02.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
delay = int(1000 / fps)
fourcc = cv2.VideoWriter.fourcc(*'mp4v')
out = cv2.VideoWriter('output_02.mp4',fourcc,30,(1280,720)) 

DROWSY_FRAMES = int(fps * 0.5)

In [6]:
beep_played = False

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (1280, 720))
    h, w = frame.shape[:2]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    
    if results.multi_face_landmarks:

        landmarks = results.multi_face_landmarks[0].landmark
        # LEFT EYE LANDMARKS
        left_eye = []
        for i in LEFT_EYE:
            x = int(landmarks[i].x * w)
            y = int(landmarks[i].y * h)
            left_eye.append((x, y))
        left_eye = np.array(left_eye)
        # RIGHT EYE LANDMARKS 
        right_eye = []
        for i in RIGHT_EYE:
            x = int(landmarks[i].x * w)
            y = int(landmarks[i].y * h)
            right_eye.append((x, y))
        right_eye = np.array(right_eye)

        left_ear = eye_aspect_ratio(left_eye)
        right_ear = eye_aspect_ratio(right_eye)
        ear = (left_ear + right_ear) / 2.0
        
        cv2.polylines(frame, [left_eye], True, (0, 255, 0), 1)
        cv2.polylines(frame, [right_eye], True, (0, 255, 0), 1)
        
        if ear < EAR_THRESHOLD:
            closed_frames += 1
        else:
            closed_frames = 0
            beep_played = False 

        if closed_frames >= DROWSY_FRAMES:
            cv2.putText(frame,"DROWSINESS ALERT",(400, 80),cv2.FONT_HERSHEY_SIMPLEX,1.5,(0, 0, 255),4)
            if not beep_played:
                winsound.Beep(1000, 700)  
                beep_played = True

        cv2.putText(frame,f"EAR: {ear:.2f}",(30, 40),cv2.FONT_HERSHEY_SIMPLEX,0.9,(255, 255, 0),2)
    
    cv2.imshow("Drowsiness Detection", frame)
    out.write(frame)
    if cv2.waitKey(delay) & 0xFF == ord('q'):  
        break

cap.release()
cv2.destroyAllWindows()

c:\Users\ASUS\Desktop\LEARN AI ML DL\DL_Projects\MEDIA-PIPELINE-PROJECT\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
